# E-commerce Sales Exploratory Data Analysis (EDA)

**Goal:** analyze online retail transactions to understand revenue trends, customer behavior, product performance, and data quality.

Dataset: UCI Machine Learning Repository — Online Retail. The notebook downloads the source Excel file when run.

## 1. Business questions
- How does revenue change over time?
- Which products and countries contribute the most revenue?
- When are sales busiest?
- How concentrated is revenue among customers?
- What data-quality issues need to be handled?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs/charts')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the dataset
The source file is downloaded from UCI if it is not already present in `data/`.

In [ ]:
dataset_path = DATA_DIR / 'online_retail.xlsx'
dataset_url = 'https://archive.ics.uci.edu/static/public/352/online+retail.zip'
zip_path = DATA_DIR / 'online_retail.zip'

if not dataset_path.exists():
    print('Downloading the UCI Online Retail dataset...')
    urllib.request.urlretrieve(dataset_url, zip_path)
    import zipfile
    with zipfile.ZipFile(zip_path) as archive:
        xlsx_name = next(name for name in archive.namelist() if name.lower().endswith('.xlsx'))
        archive.extract(xlsx_name, DATA_DIR)
        extracted = DATA_DIR / xlsx_name
        if extracted != dataset_path:
            extracted.replace(dataset_path)
    zip_path.unlink(missing_ok=True)

df = pd.read_excel(dataset_path)
print('Dataset shape:', df.shape)
display(df.head())

## 3. Initial inspection and data quality

In [ ]:
display(df.info())
display(df.describe(include='all').T)
quality = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2),
    'unique_values': df.nunique(dropna=True)
}).sort_values('missing_percent', ascending=False)
display(quality)
print('Duplicate rows:', df.duplicated().sum())

## 4. Clean and prepare data
Keep the original dataframe unchanged. Exclude rows without product descriptions or customer IDs for analyses that require those fields. Treat cancellations separately using invoice codes beginning with `C`.

In [ ]:
clean = df.drop_duplicates().copy()
clean['InvoiceDate'] = pd.to_datetime(clean['InvoiceDate'], errors='coerce')
clean['Quantity'] = pd.to_numeric(clean['Quantity'], errors='coerce')
clean['UnitPrice'] = pd.to_numeric(clean['UnitPrice'], errors='coerce')
clean['CustomerID'] = clean['CustomerID'].astype('Int64')
clean = clean.dropna(subset=['InvoiceNo', 'InvoiceDate', 'Quantity', 'UnitPrice'])
clean['InvoiceNo'] = clean['InvoiceNo'].astype(str)
clean['IsCancellation'] = clean['InvoiceNo'].str.startswith('C')
clean['Revenue'] = clean['Quantity'] * clean['UnitPrice']
clean['YearMonth'] = clean['InvoiceDate'].dt.to_period('M').astype(str)
clean['Month'] = clean['InvoiceDate'].dt.month_name()
clean['DayOfWeek'] = clean['InvoiceDate'].dt.day_name()
clean['Hour'] = clean['InvoiceDate'].dt.hour
sales = clean.loc[(~clean['IsCancellation']) & (clean['Quantity'] > 0) & (clean['UnitPrice'] > 0)].copy()
identified_sales = sales.dropna(subset=['CustomerID']).copy()
print('Original rows:', len(df))
print('Rows after deduplication and required-field checks:', len(clean))
print('Valid positive sales rows:', len(sales))
print('Sales rows with customer IDs:', len(identified_sales))
display(sales.head())

## 5. Key performance indicators

In [ ]:
kpis = pd.Series({
    'Total revenue': sales['Revenue'].sum(),
    'Transactions': sales['InvoiceNo'].nunique(),
    'Units sold': sales['Quantity'].sum(),
    'Unique products': sales['StockCode'].nunique(),
    'Countries served': sales['Country'].nunique(),
    'Identified customers': identified_sales['CustomerID'].nunique(),
    'Average order value': sales.groupby('InvoiceNo')['Revenue'].sum().mean()
})
display(kpis.to_frame('Value').style.format('{:,.2f}'))

## 6. Monthly revenue trend

In [ ]:
monthly_revenue = sales.groupby('YearMonth')['Revenue'].sum().sort_index()
ax = monthly_revenue.plot(kind='line', figsize=(12, 5), marker='o')
ax.set(title='Monthly Revenue Trend', xlabel='Month', ylabel='Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
display(monthly_revenue.to_frame('Revenue').tail(12))

## 7. Best-performing products

In [ ]:
top_products = (sales.groupby('Description', dropna=True)['Revenue'].sum()
                .sort_values(ascending=False).head(10).sort_values())
ax = top_products.plot(kind='barh', figsize=(10, 6))
ax.set(title='Top 10 Products by Revenue', xlabel='Revenue', ylabel='Product')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'top_products.png', dpi=150, bbox_inches='tight')
plt.show()
display(top_products.sort_values(ascending=False).to_frame('Revenue'))

## 8. Country-level sales

In [ ]:
country_sales = sales.groupby('Country').agg(
    revenue=('Revenue', 'sum'),
    orders=('InvoiceNo', 'nunique'),
    units=('Quantity', 'sum')
).sort_values('revenue', ascending=False)
display(country_sales.head(10))
ax = country_sales['revenue'].head(10).sort_values().plot(kind='barh', figsize=(9, 5))
ax.set(title='Top 10 Countries by Revenue', xlabel='Revenue', ylabel='Country')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'country_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Sales by weekday and hour

In [ ]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_sales = sales.groupby('DayOfWeek')['Revenue'].sum().reindex(weekday_order)
weekday_sales.plot(kind='bar', figsize=(9, 4))
plt.title('Revenue by Day of Week')
plt.xlabel('Day')
plt.ylabel('Revenue')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'weekday_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

hourly_sales = sales.groupby('Hour')['Revenue'].sum()
hourly_sales.plot(kind='bar', figsize=(9, 4))
plt.title('Revenue by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Revenue')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hourly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Customer analysis

In [ ]:
customer_sales = identified_sales.groupby('CustomerID').agg(
    revenue=('Revenue', 'sum'),
    orders=('InvoiceNo', 'nunique'),
    units=('Quantity', 'sum')
).sort_values('revenue', ascending=False)
display(customer_sales.head(10))
customer_sales['revenue'].plot(kind='hist', bins=50, figsize=(9, 4))
plt.title('Distribution of Customer Revenue')
plt.xlabel('Customer revenue')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'customer_revenue_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
top_10_share = customer_sales['revenue'].head(10).sum() / customer_sales['revenue'].sum() * 100
print(f'Top 10 customers contribute {top_10_share:.2f}% of identified-customer revenue.')

## 11. Cancellation analysis

In [ ]:
cancellations = clean.loc[clean['IsCancellation']].copy()
print('Cancellation invoice count:', cancellations['InvoiceNo'].nunique())
print('Cancellation line count:', len(cancellations))
print('Cancellation line value (absolute):', f"{(cancellations['Quantity'] * cancellations['UnitPrice']).abs().sum():,.2f}")
display(cancellations.head())

## 12. Export cleaned data and summary tables

In [ ]:
clean.to_csv(DATA_DIR / 'online_retail_cleaned.csv', index=False)
monthly_revenue.rename('Revenue').to_csv(OUTPUT_DIR / 'monthly_revenue.csv')
country_sales.to_csv(OUTPUT_DIR / 'country_sales.csv')
customer_sales.to_csv(OUTPUT_DIR / 'customer_sales.csv')
print('Exported cleaned data and summary tables.')

## 13. Findings and recommendations
After running the notebook, record evidence-backed findings here. Avoid inventing conclusions before checking the outputs.

Suggested questions for your final write-up:
1. Which month had the highest revenue, and was it a complete month?
2. How much revenue comes from the largest country and top products?
3. Which weekday and hour appear strongest?
4. How concentrated is customer revenue?
5. What are the main data-quality limitations?
6. What business actions would you recommend based on the evidence?